In [1]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("spam-experiment")

<Experiment: artifact_location='/mnt/c/Users/user/Documents/Projects/mlops-exer-1/notebooks/mlruns/1', creation_time=1780963326545, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780963326545, lifecycle_stage='active', name='spam-experiment', tags={}, trace_location=None, workspace='default'>

In [2]:
# from spam_checker.data.spam_data_module import SPAMDataModule

In [3]:
# spam_data = SPAMDataModule()

In [4]:
# spam_data.prepare_data()

In [5]:
import pandas as pd

# Paste your copied file path inside the quotes
# df = pd.read_csv('../test_dataset/spam.csv')

df = pd.read_csv('../test_dataset/spam.csv', encoding_errors='ignore')

df = df[['v1', 'v2']]

# df['v1'] = df['v1'].replace('spam', '1')
# df['v1'] = df['v1'].replace('ham', '0')
df = df.rename(columns={"v1": "label", "v2": "message"})

# View the first few rows of your data
df.head(10)

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
5,spam,FreeMsg Hey there darling it's been 3 week's n...
6,ham,Even my brother is not like to speak with me. ...
7,ham,As per your request 'Melle Melle (Oru Minnamin...
8,spam,WINNER!! As a valued network customer you have...
9,spam,Had your mobile 11 months or more? U R entitle...


In [6]:
df['label'] = df['label'].replace('spam', '1')
df['label'] = df['label'].replace('ham', '0')

# View the first few rows of your data
df.head(10)

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [7]:
len(df.index)

5572

In [8]:
empty_spam_rows = df[df['message'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, message]
Index: []


In [9]:
empty_count = df['message'].isna().sum()
print(f"Empty cells: {empty_count}")

Empty cells: 0


In [10]:
# !python -m spacy download en_core_web_sm

In [11]:
# import spacy

# nlp = spacy.load("en_core_web_sm")

In [12]:
# doc = nlp("spaCy is successfully working in Jupyter!")
# for token in doc:
#     print(token.text, token.pos_)

In [13]:
# # Function to clean the text
# def clean_text(text):
#     doc = nlp(text)  # Process the text with spaCy
#     cleaned_tokens = []
#     for token in doc:
#         # Remove stop words, punctuation, and retain only alphabetic words
#         if not token.is_stop and not token.is_punct and token.is_alpha:
#             cleaned_tokens.append(token.lemma_)  # Append the lemmatized version of the token
#     return " ".join(cleaned_tokens)

# Apply the cleaning function to the 'text' column in the dataset
# df['cleaned_text'] = df['text'].apply(clean_text)

In [14]:
df.head(10)

,label,message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
5,1,FreeMsg Hey there darling it's been 3 week's n...
6,0,Even my brother is not like to speak with me. ...
7,0,As per your request 'Melle Melle (Oru Minnamin...
8,1,WINNER!! As a valued network customer you have...
9,1,Had your mobile 11 months or more? U R entitle...


In [15]:
training_input_labels = df.loc[:, df.columns == 'label']
training_input_text = df.loc[:, df.columns != 'label']

print(training_input_labels['label'].value_counts())

label
0    4825
1     747
Name: count, dtype: int64


In [16]:
print(training_input_labels)

     label
0        0
1        0
2        1
3        0
4        0
...    ...
5567     1
5568     0
5569     0
5570     0
5571     0

[5572 rows x 1 columns]


In [17]:
from sklearn.model_selection import train_test_split

train_text_temp, val_text_final, train_labels_temp, val_labels = train_test_split(
    training_input_text, training_input_labels, train_size=0.5, random_state=0, stratify=training_input_labels)

train_text_final, test_text_final, train_labels, test_labels = train_test_split(
    train_text_temp, train_labels_temp, train_size=0.8, random_state=0, stratify=train_labels_temp)


In [18]:
train_text_combined = pd.concat([train_labels, train_text_final], axis=1).reset_index(drop=True)

In [19]:
test_text_combined = pd.concat([test_labels, test_text_final], axis=1).reset_index(drop=True)

In [20]:
val_text_combined = pd.concat([val_labels, val_text_final], axis=1).reset_index(drop=True)

In [21]:
empty_spam_rows = train_text_combined[train_text_combined['message'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, message]
Index: []


In [22]:
# empty_spam_rows = train_text_combined[train_text_combined['cleaned_text'].isna()]
# print(empty_spam_rows)

In [23]:
empty_spam_rows = train_text_combined[train_text_combined['label'].isna()]
print(empty_spam_rows)

Empty DataFrame
Columns: [label, message]
Index: []


In [24]:
# def spacy_tokenizer(text):
#     # Use the clean_lemm function first
#     cleaned_text = clean_text(text)
#     # Then tokenize the cleaned text
#     return cleaned_text.split()

In [25]:
# from collections import defaultdict

# def build_vocab(text_iterator, min_freq=3, specials=('<unk>', '<pad>')):
#     token_counts = defaultdict(int)
#     for text in text_iterator:
#         for token in spacy_tokenizer(text):
#             token_counts[token] += 1
#     vocab = {token: idx for idx, (token, count) in enumerate(token_counts.items()) if count >= min_freq}
#     for special in specials:
#         if special not in vocab:
#             vocab[special] = len(vocab) 
#     return vocab
# vocab = build_vocab(df['text'])
# # vocab = build_vocab(df['cleaned_text'])
# vocab_size = len(vocab)

In [26]:
# print(vocab_size)

In [27]:
# print(vocab)

In [28]:
# sample data
#   label  message
# 0 ham    Hey, are we still meeting today?
# 1 spam   WIN a FREE vacation now!!!

In [ ]:
# %run training/run_training.py --data='../test_dataset/spam.csv' \
# --name_of_label_column='v1' \
# --name_of_message_column='v2' \
# --num_workers=0

# %run training/run_training.py --data='../test_dataset/spam.csv' \
# --name_of_label_column='v1' \
# --name_of_message_column='v2' \
# --max_epochs=10

# import sys
# import os
# from pathlib import Path

# sys.path.append(str(Path.cwd().parent))

# !python -m training/run_training --data='../test_dataset/spam.csv' \
# --name_of_label_column='v1' \
# --name_of_message_column='v2' \
# --num_workers=0

call main
calling smsdataset
calling smsdataset
calling smsdataset
<------------------------data load complete-------------------------------->
<------------------------model load complete-------------------------------->


/mnt/c/Users/user/Documents/Projects/mlops-exer-1/notebooks/spam_checker/data/spam_lit_datamodule.py:57: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"ham": 0, "spam": 1})
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of t

<------------------------trainer build complete-------------------------------->


/mnt/c/Users/user/Documents/Projects/mlops-exer-1/notebooks/spam_checker/data/spam_lit_datamodule.py:57: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({"ham": 0, "spam": 1})
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


calling smsdataset
calling smsdataset
calling smsdataset


...

/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/carl/.virtualenvs/mlops-exer-1/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=3` in the `DataLoader` to improve performance.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.9874551892280579     │
│         test_loss         │    0.06031934544444084    │
└───────────────────────────┴───────────────────────────┘

<------------------------trainer fit complete-------------------------------->
calling smsdataset
calling smsdataset
calling smsdataset
running test dataloader


`weights_only` was not set, defaulting to `False`.


<------------------------trainer test complete-------------------------------->


In [30]:
# from spam_checker.data.spam_lit_datamodule import SMSDataModule

# data = SMSDataModule(
#     dataframe=df,
#     batch_size=64,
#     # num_workers=1
# )

# data.setup()

# print(len(data.vocab))

In [31]:
# print(len(data.test_dataset))

In [32]:
# print(data.test_dataset)

In [33]:
# from spam_checker.models.spam_classifier import SpamClassifier

# model = SpamClassifier(
#     vocab_size=len(data.vocab)
# )

In [34]:
# from lightning import Trainer
# trainer = Trainer(max_epochs=10, accelerator="auto", devices="auto")



In [35]:
# trainer.fit(
#     model,
#     datamodule=data,
# )

In [36]:
# trainer.test(
#     model,
#     datamodule=data,
# )

In [37]:
# result = predict_sms(
#     "Congratulations! You have won a free iPhone. Click here now!",
#     model,
#     data.vocab,
# )

# print(result)

In [38]:
# import torch

# torch.save(data.vocab, "vocab.pt")
# trainer.save_checkpoint("sms_spam.ckpt")

In [ ]:
from spam_checker.models.spam_classifier import SpamClassifier
import torch

vocab = torch.load("../vocab.pt")

model = SpamClassifier.load_from_checkpoint(
    "../sms_spam.ckpt",
    vocab_size=len(vocab),
)

In [40]:
from spam_checker.predict_sms import predict_sms
result = predict_sms(
    "Congratulations! You have won a free iPhone. Click here now!",
    model,
    vocab,
)

print(result)

{'prediction': 'spam', 'spam_probability': 0.9373047947883606}


In [41]:
result = predict_sms(
    "U dun say so early hor... U c already then say...",
    model,
    vocab,
)

print(result)

{'prediction': 'ham', 'spam_probability': 0.00023737893206998706}


In [42]:
result = predict_sms(
    "Hi. Wk been ok - on hols now! Yes on for a bit of a run. Forgot that i have hairdressers appointment at four so need to get home n shower beforehand. Does that cause prob for u?",
    model,
    vocab,
)

print(result)

{'prediction': 'ham', 'spam_probability': 7.051489228615537e-05}


In [43]:
result = predict_sms(
    "Please call our customer service representative on 0800 169 6031 between 10am-9pm as you have WON a guaranteed å£1000 cash or å£5000 prize!",
    model,
    vocab,
)

print(result)

{'prediction': 'spam', 'spam_probability': 0.9995539784431458}
